# STAQ — with Remedy 1: Type-Gated Spatial Smoothness Loss

This notebook implements **Remedy 1 (A1)** from the professor's methodology report
*"Achieving Cell–Type Homogeneity in Spatial Metacells"*:

> Replace the unconditional spatial smoothness penalty with a **gated** penalty,
> where each spatial edge is weighted by transcriptomic agreement between its two
> endpoints, computed from a **spatially-blind** embedding (Stage 1 PCA — never the
> GAT output, to avoid circularity).

**Only this one change is made.** Encoder, codebook, decoder, Stage 5 anchors,
Stage 7 contiguity refinement, and Stage 8 aggregation are all unchanged from the
baseline pipeline.

Pipeline: Stage 1 → 2 → 3 → 4 (+ z_pca for the gate) → 5 → 6 (gated loss) → 7 → 8 → 9 → plots.


In [ ]:
import subprocess, sys

packages = [
    "numpy==2.0.2",
    "scipy==1.15.3",
    "zarr==2.18.3",
    "anndata==0.10.9",
    "scanpy==1.10.3",
    "torch-geometric==2.6.1",
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + packages,
    check=True
)

print("ALL DONE — restart kernel now")


## Stage 1 — Preprocessing

In [ ]:
# =================================================================
# STAGE 1 — PREPROCESSING  (MERFISH Motor Cortex — COVET paper)
# =================================================================
# Dataset: Mouse Primary Motor Cortex MERFISH
# Source:  Pe'er lab AWS — dp-lab-data-public S3 bucket
#          (official ENVI/COVET tutorial dataset, Nat Biotech 2024)
# File:    st_data.h5ad
# Cells:   ~276,556 total across multiple slices
# Genes:   254
# Cell type column: obs["cell_type"]  (23 cell types)
# =================================================================

import os
import scanpy as sc
import scipy.sparse as sp
import numpy as np
import warnings
warnings.filterwarnings("ignore")

SAVE_DIR = "/kaggle/working"
os.makedirs(SAVE_DIR, exist_ok=True)
CT_COL   = "cell_type"

print("=" * 60)
print("STAGE 1 — MERFISH Motor Cortex Preprocessing")
print("=" * 60)

# ── 1. Load data ─────────────────────────────────────────────────
# change this path to wherever you uploaded st_data.h5ad on Kaggle
DATA_PATH = "/kaggle/input/datasets/vishnutavanam2709/cortex/st_data.h5ad"

print(f"loading from: {DATA_PATH}")
adata_full = sc.read_h5ad(DATA_PATH)
print(f"full dataset  : {adata_full.n_obs:,} cells x {adata_full.n_vars} genes")

# ── 2. Show available slices ─────────────────────────────────────
slice_counts = adata_full.obs["batch"].value_counts().sort_index()
print()
print("cells per slice:")
for s, c in slice_counts.items():
    print(f"  {s}: {c:,} cells")

# ── 3. Pick one slice ────────────────────────────────────────────
# mouse1_slice10 is the slice used in the ENVI tutorial
SLICE_ID = "mouse1_slice170"

adata = adata_full[adata_full.obs["batch"] == SLICE_ID].copy()
print(f"\nusing slice   : {SLICE_ID}")
print(f"slice loaded  : {adata.n_obs:,} cells x {adata.n_vars} genes")

# ── 4. Remove blank probe genes ──────────────────────────────────
n_blank = adata.var_names.str.startswith("Blank").sum()
adata   = adata[:, ~adata.var_names.str.startswith("Blank")].copy()
print(f"removed {n_blank} blank probes -> {adata.n_vars} genes remain")

# ── 5. Densify and save raw counts ───────────────────────────────
if sp.issparse(adata.X):
    adata.X = adata.X.toarray().astype("float32")
adata.layers["counts"] = adata.X.copy()

# ── 6. Library-size normalisation + log2 transform ───────────────
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata, base=2)
adata.layers["log_counts"] = adata.X.copy()

# ── 7. PCA ───────────────────────────────────────────────────────
sc.tl.pca(adata, n_comps=30)

# ── 8. Save ──────────────────────────────────────────────────────
adata.write_h5ad(os.path.join(SAVE_DIR, "stage1.h5ad"))

print()
print("OUTPUT — stage1.h5ad:")
print(f"  n_obs      : {adata.n_obs:,} cells")
print(f"  n_vars     : {adata.n_vars} genes")
print(f"  X_pca      : {adata.obsm['X_pca'].shape}")
print(f"  spatial    : {adata.obsm['spatial'].shape}")
print(f"  slice      : {SLICE_ID}")
print(f"  cell types : {sorted(adata.obs[CT_COL].unique())}")
print("[SUCCESS] Stage 1 complete")


## Stage 2 — Dual-view kNN graphs (W_T, W_S)

In [ ]:
# =================================================================
# STAGE 2 — DUAL-VIEW kNN GRAPHS
# =================================================================
# Builds W_T (transcriptomic, PCA space) and W_S (spatial, coordinate
# space) — both k=15, adaptive-bandwidth Gaussian kernel, symmetrised.
#
# W_S is the graph actually used for GAT message passing (Stage 6),
# the spatial smoothness loss (Stage 6), the niche profile (Stage 3)
# and contiguity refinement (Stage 7).
# =================================================================

import os
import numpy as np
import anndata
import scipy.sparse as sp
from sklearn.neighbors import NearestNeighbors
import warnings
warnings.filterwarnings("ignore")

SAVE_DIR = "/kaggle/working"
print("=" * 60)
print("STAGE 2 — Dual-view kNN Graphs")
print("=" * 60)

adata = anndata.read_h5ad(os.path.join(SAVE_DIR, "stage1.h5ad"))
Z = adata.obsm["X_pca"].astype("float32")     # (n, 30)
S = adata.obsm["spatial"].astype("float32")   # (n, 2)
n = Z.shape[0]
print(f"n = {n}   Z shape = {Z.shape}   S shape = {S.shape}")

def build_knn_graph(features, k, name):
    print(f"  building {name}  (k={k}) ...")
    nbrs = NearestNeighbors(n_neighbors=k + 1, n_jobs=-1)
    nbrs.fit(features)
    distances, indices = nbrs.kneighbors(features)
    sigma = distances[:, k // 2].copy()
    sigma[sigma == 0] = 1e-8
    src = np.repeat(np.arange(n), k)
    dst = indices[:, 1:].flatten()
    d   = distances[:, 1:].flatten()
    w   = np.exp(-(d ** 2) / (sigma[src] * sigma[dst]))
    W   = sp.csr_matrix((w, (src, dst)), shape=(n, n))
    W   = (W + W.T) * 0.5
    W.eliminate_zeros()
    print(f"    edges = {W.nnz:,}   density = {W.nnz / (n * n) * 100:.3f}%")
    return W

W_T = build_knn_graph(Z, k=15, name="W_T  (transcriptomic, PCA space)")
W_S = build_knn_graph(S, k=15, name="W_S  (spatial, coordinate space)")

sp.save_npz(os.path.join(SAVE_DIR, "stage2_WT.npz"), W_T)
sp.save_npz(os.path.join(SAVE_DIR, "stage2_WS.npz"), W_S)

print()
print("OUTPUT:")
print(f"  stage2_WT.npz   shape={W_T.shape}   {W_T.nnz:,} edges")
print(f"  stage2_WS.npz   shape={W_S.shape}   {W_S.nnz:,} edges")
print()
print("[SUCCESS] stage2_WT.npz, stage2_WS.npz saved")


## Stage 3 — Niche profile

In [ ]:
# =================================================================
# STAGE 3 — NICHE PROFILE   N = D_S^-1 . W_S . Z
# =================================================================
# Frozen, non-learned neighbourhood fingerprint per cell, concatenated
# into the encoder input in Stage 4. Unchanged by Remedy 1.
# =================================================================

import os
import numpy as np
import anndata
import scipy.sparse as sp
import warnings
warnings.filterwarnings("ignore")

SAVE_DIR = "/kaggle/working"
print("=" * 60)
print("STAGE 3 — Niche Profile   N = D_S^-1 . W_S . Z")
print("=" * 60)

adata = anndata.read_h5ad(os.path.join(SAVE_DIR, "stage1.h5ad"))
W_S   = sp.load_npz(os.path.join(SAVE_DIR, "stage2_WS.npz"))
Z     = adata.obsm["X_pca"].astype("float32")   # (n, 30)

print(f"W_S shape = {W_S.shape}    Z shape = {Z.shape}")

deg           = np.array(W_S.sum(axis=1)).ravel()
deg[deg == 0] = 1.0
D_inv         = sp.diags(1.0 / deg)
W_norm        = D_inv.dot(W_S)
N             = W_norm.dot(Z).astype("float32")    # (n, 30)

print(f"niche profile N shape = {N.shape}")
print(f"  Z : mean={Z.mean():.4f}  std={Z.std():.4f}")
print(f"  N : mean={N.mean():.4f}  std={N.std():.4f}")

np.save(os.path.join(SAVE_DIR, "stage3_niche.npy"), N)

print()
print("OUTPUT:")
print(f"  stage3_niche.npy   shape={N.shape}")
print()
print("[SUCCESS] stage3_niche.npy saved")


## Stage 4 — Tensor assembly (niche profile N removed from input)

**Changed.** `x_input` is now just `log_counts` on its own — the
niche profile `N` (Stage 3's neighbour-averaged expression) is no
longer concatenated in. Stage 3 still computes and saves it, it's
just not fed into the model anymore. This directly tests whether
baking a hand-computed niche signal into the encoder's input was
adding noise on top of what the GAT branch (`c_niche`) already
computes for itself through attention over the spatial graph.

In [ ]:
# =================================================================
# STAGE 4 — TENSOR ASSEMBLY   (niche profile N REMOVED from x_input)
# =================================================================
# CHANGED — x_input is now JUST log_counts, no niche profile
# concatenated on. The niche profile N (Stage 3's neighbour-averaged
# expression) is still computed by Stage 3, but no longer fed into
# the model at all -- this tests whether baking a hand-computed
# niche signal into the encoder's input was actually hurting results,
# given the GAT branch (c_niche) already computes its own
# neighbourhood signal via attention, making N's explicit inclusion
# potentially redundant noise on top of that.
#
# x_input  = log_counts(156)                      = (n, 156)   encoder input
# y_target = raw counts                            = (n, 156)   NB target
# z_pca    = Stage 1 PCA embedding                  = (n, 30)
#            spatially-blind, used ONLY to compute the gate w_ij
#            in the Remedy 1 gated spatial loss (Stage 6).
#            Never touched by the encoder, never gets gradients.
# =================================================================

import os
import numpy as np
import anndata
import scipy.sparse as sp
import torch
import warnings
warnings.filterwarnings("ignore")

SAVE_DIR = "/kaggle/working"
print("=" * 60)
print("STAGE 4 — Tensor Assembly")
print("=" * 60)

adata = anndata.read_h5ad(os.path.join(SAVE_DIR, "stage1.h5ad"))
W_T   = sp.load_npz(os.path.join(SAVE_DIR, "stage2_WT.npz"))
W_S   = sp.load_npz(os.path.join(SAVE_DIR, "stage2_WS.npz"))
# NOTE: stage3_niche.npy (N) is intentionally NOT loaded here anymore --
# Stage 3 still computes and saves it, but it no longer feeds the model.

x_log   = adata.layers["log_counts"].astype("float32")   # (n, 156)
x_input = x_log.copy()                                     # CHANGED — no niche concat
print(f"x_input  = log_counts only (niche N removed) = {x_input.shape}")

raw = adata.layers["counts"]
if sp.issparse(raw):
    raw = raw.toarray()
raw = raw.astype("float32")
print(f"y_target = raw counts  {raw.shape}   sum={raw.sum():.0f}")

# spatially-blind PCA embedding — the SOFT gate source (fallback) for Remedy 1
z_pca = adata.obsm["X_pca"].astype("float32")   # (n, 30)
print(f"z_pca    = Stage 1 PCA (spatially blind, soft-gate fallback source)  {z_pca.shape}")

# integer-encoded cell type — the HARD gate source (default) for Remedy 1
CT_COL = "cell_type"
if CT_COL in adata.obs.columns:
    ct_categorical = adata.obs[CT_COL].astype("category")
    cell_type_idx  = ct_categorical.cat.codes.values.astype("int64")   # (n,)
    ct_classes     = list(ct_categorical.cat.categories)
    print(f"cell_type_idx = integer-coded '{CT_COL}'  {cell_type_idx.shape}  "
          f"({len(ct_classes)} classes: {ct_classes})")
else:
    cell_type_idx = np.full(x_input.shape[0], -1, dtype="int64")   # sentinel: no labels
    print(f"no '{CT_COL}' column found — cell_type_idx filled with -1 "
          f"(Stage 6 will fall back to the soft gate)")

def to_edge_tensors(W, name):
    W_coo = W.tocoo()
    ei = torch.tensor(np.vstack([W_coo.row, W_coo.col]), dtype=torch.long)
    ew = torch.tensor(W_coo.data, dtype=torch.float32)
    print(f"  {name}:  edge_index={ei.shape}  edge_weight={ew.shape}")
    return ei, ew

ei_S, ew_S = to_edge_tensors(W_S, "W_S (spatial)")
ei_T, ew_T = to_edge_tensors(W_T, "W_T (transcriptomic)")

tensors = {
    "x_input":       torch.tensor(x_input,       dtype=torch.float32),
    "y_target":      torch.tensor(raw,           dtype=torch.float32),
    "z_pca":         torch.tensor(z_pca,         dtype=torch.float32),   # NEW — soft gate fallback
    "cell_type_idx": torch.tensor(cell_type_idx, dtype=torch.long),      # NEW — hard gate (default)
    "edge_index_S":  ei_S,
    "edge_weight_S": ew_S,
    "edge_index_T":  ei_T,
    "edge_weight_T": ew_T,
}

torch.save(tensors, os.path.join(SAVE_DIR, "stage4_tensors.pt"))

print()
print("OUTPUT — stage4_tensors.pt:")
for k, v in tensors.items():
    print(f"  '{k}':  shape={tuple(v.shape)}   dtype={v.dtype}")
print()
print("[SUCCESS] stage4_tensors.pt saved")

## Stage 5 — Codebook initialisation (Stratified FPS) — unchanged

In [ ]:
# =================================================================
# STAGE 5 — CODEBOOK INITIALISATION  (Stratified FPS)
# =================================================================
# Unchanged by Remedy 1. Picks M anchor cells spread across both PCA
# space and spatial space, used to seed the codebook in Stage 6.
# =================================================================

import os
import numpy as np
import anndata
import warnings
warnings.filterwarnings("ignore")

SAVE_DIR = "/kaggle/working"
print("=" * 60)
print("STAGE 5 — Codebook Init  (Stratified FPS)")
print("=" * 60)

adata = anndata.read_h5ad(os.path.join(SAVE_DIR, "stage1.h5ad"))
Z = adata.obsm["X_pca"].astype("float32")    # (n, 30)
S = adata.obsm["spatial"].astype("float32")  # (n, 2)
n = Z.shape[0]

gamma_M = 75
M       = ceil({n}/{gamma_M})
beta    = 0.5
print(f"n = {n}   M = ceil({n}/{gamma_M}) = {M}   beta = {beta}")

rng    = np.random.default_rng(42)
sample = rng.choice(n, min(2000, n), replace=False)
med_T  = float(np.median(np.linalg.norm(
    Z[sample][:, None] - Z[sample][None, :], axis=-1)))
med_S  = float(np.median(np.linalg.norm(
    S[sample][:, None] - S[sample][None, :], axis=-1)))
print(f"median dist — transcriptomic: {med_T:.4f}   spatial: {med_S:.4f}")

seed     = int(np.argmax(np.linalg.norm(Z - Z.mean(axis=0), axis=1)))
selected = [seed]
min_dT   = np.linalg.norm(Z - Z[seed], axis=1)
min_dS   = np.linalg.norm(S - S[seed], axis=1)

print(f"running FPS for M={M} anchors ...")
for step in range(2, M + 1):
    rho            = beta * (min_dT / med_T) + (1 - beta) * (min_dS / med_S)
    rho[selected]  = -np.inf
    i_star         = int(np.argmax(rho))
    selected.append(i_star)
    min_dT = np.minimum(min_dT, np.linalg.norm(Z - Z[i_star], axis=1))
    min_dS = np.minimum(min_dS, np.linalg.norm(S - S[i_star], axis=1))
    if step % 200 == 0:
        print(f"  step {step}/{M} done")

anchor_idx = np.array(selected)
np.save(os.path.join(SAVE_DIR, "stage5_anchors.npy"), anchor_idx)

print()
print("OUTPUT:")
print(f"  stage5_anchors.npy   shape={anchor_idx.shape}   ({M} anchor cell indices)")
print()
print("[SUCCESS] stage5_anchors.npy saved")


## Stage 6 — Full STAQ training, with Remedy 1 (gated spatial smoothness loss) + Remedy 4 (two-branch encoder, FiLM decoder)

**Architecture change from the previous version:** Remedy 2's soft
`LAMBDA` blend has been replaced by Remedy 4's hard architectural
split — `z_cell` (spatially blind, feeds the codebook) and `c_niche`
(neighbourhood-aware, feeds the decoder via FiLM) are now genuinely
separate tensors, never combined into one embedding.

**Remedy 1 (the gated spatial loss) is kept, unchanged in its own
logic** — it still gates on `cell_type_idx` (hard mode) or `z_pca`
distance (soft mode) exactly as before. What changes is *what it's
now safe to say about it*: since `p_sp` is now computed purely from
`z_cell` — which cannot see neighbours at all — this loss can only
ever pull already-same-type spatial neighbours toward the same
codebook entry. There are now two independent reasons cross-type
merging cannot happen through this pathway: the gate itself, and
`z_cell`'s architectural blindness. This combination was verified
piece-by-piece on a small synthetic toy graph before being wired in
here: `z_cell` was confirmed identical regardless of which neighbours
a cell was given, `c_niche` was confirmed to change with the graph,
and the coldbook assignment was confirmed to be completely unaffected
by which niche embedding was passed to the decoder.

Everything else — Stage 4/5 tensors, `get_mini_batch`, `fix_codebook`,
Stage 7/8/9 — is unchanged; this is an encoder/decoder-only
architectural swap.

In [ ]:
# =================================================================
#  FULL STAQ  (Two-Branch Encoder + FiLM Decoder [Remedy 4]
#              + VQ + GATED Spatial [Remedy 1] + Usage)
# =================================================================
# REMEDY 1 (A1) — type-gated spatial smoothness loss  (KEPT)
#   L_spatial_gated = sum_(i,j) W_S[i,j] * w_ij * JSD(p_sp_i, p_sp_j)
#
#   GATE_MODE = "hard"  (default, used here):
#       w_ij = 1[ cell_type[i] == cell_type[j] ]      (PDF eq. iii)
#
#   GATE_MODE = "soft"  (fallback if no labels / for ablation):
#       w_ij = exp(-||PCA_i - PCA_j||^2 / (2*sigma^2))
#
#   IMPORTANT — why this is still safe (in fact stronger) under Remedy 4:
#   p_sp now comes from distances between z_cell (the SPATIALLY BLIND
#   branch, see below) and the codebook. Since z_cell architecturally
#   cannot encode which cell type a cell is NOT, this loss can only
#   ever pull already-same-type spatial neighbours closer together in
#   codebook-assignment space (helping spatial compactness). Gate=0
#   additionally blocks it from ever touching cross-type pairs. So
#   there are now TWO independent reasons this loss cannot cause
#   cross-type merging: the gate, AND z_cell's structural blindness.
#
# REMEDY 4 (A4) — two-branch encoder with FiLM-conditional decoder
#   Replaces Remedy 2's soft LAMBDA blend with a HARD architectural
#   split. Two genuinely separate embeddings are produced:
#
#       z_cell  = self_fc(x_own)     -- spatially blind (plain nn.Linear,
#                                        NEVER receives ei/ew). This is
#                                        the ONLY thing the codebook
#                                        ever sees or quantises.
#       c_niche = GAT(x, ei, ew)     -- neighbourhood-aware, but NEVER
#                                        quantised. Feeds the decoder
#                                        directly via FiLM instead.
#
#   The decoder reconstructs from z_hat (quantised z_cell) with its
#   hidden activations modulated by c_niche:
#       FiLM(h | c) = gamma(c) * h + beta(c)
#
#   Because c_niche never touches the VQ lookup, cell-type purity
#   becomes a HARD GUARANTEE of the architecture rather than a tuned
#   compromise (unlike Remedy 1/2 alone) — while c_niche still lets
#   the model explain real niche-driven expression variation through
#   the decoder, so reconstruction quality isn't sacrificed for it.
# =================================================================

import os
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.nn import GATConv
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings("ignore")

# ── hyperparameters ───────────────────────────────────────────────
D      = 32
D_H    = 64
H      = 4
EPOCHS = 150
B      = 32
LR     = 1e-3
A_CB   = 0.5
A_CM   = 1.0
A_SP   = 1.0
A_UE   = 0.1
GAMMA  = 0.90
EPS    = 1e-5
N_MIN  = 25
TAU0   = 1.0
T_ANN  = EPOCHS

GATE_MODE = "hard"   # "hard" (default, uses cell_type labels) or "soft" (PCA-distance fallback)

device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAVE_DIR = "/kaggle/working"

print(f"device = {device}")
print(f"A_CB={A_CB}  A_CM={A_CM}  A_SP={A_SP}  A_UE={A_UE}")
print(f"GAMMA={GAMMA}  fixed tau_sp=1.0")
print(f"REMEDY 1 ACTIVE — gated spatial smoothness loss  (GATE_MODE='{GATE_MODE}')")
print(f"REMEDY 4 ACTIVE — two-branch encoder + FiLM-conditional decoder")
print("=" * 60)

# ── model ─────────────────────────────────────────────────────────
class TwoBranchEncoder(nn.Module):
    """Produces two SEPARATE embeddings, never blended together.
    z_cell  : spatially blind, plain nn.Linear on own expression only.
              This is the ONLY thing the VQ codebook ever sees.
    c_niche : GAT over the spatial graph, neighbourhood-aware.
              Never quantised — passed straight to the decoder's FiLM.
    """
    def __init__(self, in_dim, n_genes):
        super().__init__()
        self.n_genes = n_genes
        self.self_fc = nn.Linear(n_genes, D)     # cell branch — no ei/ew ever passed in
        self.gat1 = GATConv(in_dim, D_H, heads=H, concat=True)
        self.gat2 = GATConv(H * D_H, D, heads=H, concat=False)
        self.bn1  = nn.BatchNorm1d(H * D_H)
        self.bn2  = nn.BatchNorm1d(D)

    def forward(self, x, ei, ew):
        x_own   = x[:, :self.n_genes]                    # own log_counts only, no niche columns
        z_cell  = self.self_fc(x_own)                     # spatially blind, by construction
        h       = F.elu(self.bn1(self.gat1(x, ei, ew)))
        c_niche = F.elu(self.bn2(self.gat2(h, ei, ew)))    # neighbourhood branch
        return z_cell, c_niche

class VectorQuantizerEMA(nn.Module):
    def __init__(self, M):
        super().__init__()
        self.M = M
        self.register_buffer("codebook", torch.empty(M, D))
        self.register_buffer("N",        torch.zeros(M))
        self.register_buffer("Sigma",    torch.zeros(M, D))

    def initialize(self, E0):
        E0 = E0.to(self.codebook.device)
        self.codebook.copy_(E0)
        self.N.fill_(1.0)
        self.Sigma.copy_(E0)

    def forward(self, z, tau):
        dists = (z.pow(2).sum(1, keepdim=True)
                 + self.codebook.pow(2).sum(1)
                 - 2 * z @ self.codebook.t()).clamp(min=0)
        q     = dists.argmin(1)
        z_q   = self.codebook[q]
        z_hat = z + (z_q - z).detach()

        p     = F.softmax(-dists / tau,  dim=1)
        p_sp  = F.softmax(-dists / 1.0,  dim=1)

        l_cb  = F.mse_loss(z.detach(), z_q)
        l_cm  = F.mse_loss(z, z_q.detach())

        if self.training:
            oh = torch.zeros(z.size(0), self.M, device=z.device)
            oh.scatter_(1, q.unsqueeze(1), 1)
            self.N.mul_(GAMMA).add_(oh.sum(0),               alpha=1 - GAMMA)
            self.Sigma.mul_(GAMMA).add_(oh.t() @ z.detach(), alpha=1 - GAMMA)
            self.codebook.data.copy_(
                self.Sigma / self.N.clamp(min=EPS).unsqueeze(1))

        return z_hat, q, p, p_sp, l_cb, l_cm

class FiLM(nn.Module):
    """Feature-wise Linear Modulation: h -> gamma(c)*h + beta(c).
    c (the niche embedding) rescales and shifts a hidden layer's
    activations, per cell, per hidden unit -- it never touches z_hat
    or the codebook lookup, only the decoder's internal computation."""
    def __init__(self, cond_dim, feature_dim):
        super().__init__()
        self.gamma_fc = nn.Linear(cond_dim, feature_dim)
        self.beta_fc  = nn.Linear(cond_dim, feature_dim)

    def forward(self, h, c):
        gamma = self.gamma_fc(c)
        beta  = self.beta_fc(c)
        return gamma * h + beta

class FiLMDecoder(nn.Module):
    def __init__(self, n_genes):
        super().__init__()
        self.fc1        = nn.Linear(D, D_H)
        self.fc2        = nn.Linear(D_H, D_H)
        self.fc3        = nn.Linear(D_H, n_genes)
        self.film1      = FiLM(cond_dim=D, feature_dim=D_H)   # modulates after fc1
        self.film2      = FiLM(cond_dim=D, feature_dim=D_H)   # modulates after fc2
        self.theta_star = nn.Parameter(torch.zeros(n_genes))

    def forward(self, z_hat, c_niche, lib):
        u     = F.elu(self.fc1(z_hat))
        u     = self.film1(u, c_niche)                        # niche reshapes this layer
        u     = F.elu(self.fc2(u))
        u     = self.film2(u, c_niche)                         # and again here
        rho   = F.softmax(self.fc3(u), dim=1)
        mu    = lib.unsqueeze(1) * rho
        theta = F.softplus(self.theta_star)
        return mu, theta

class STAQ(nn.Module):
    def __init__(self, in_dim, M, n_genes):
        super().__init__()
        self.enc = TwoBranchEncoder(in_dim, n_genes)
        self.vq  = VectorQuantizerEMA(M)
        self.dec = FiLMDecoder(n_genes)

    def forward(self, x, ei, ew, lib, tau):
        z_cell, c_niche               = self.enc(x, ei, ew)
        z_hat, q, p, p_sp, l_cb, l_cm = self.vq(z_cell, tau)          # codebook sees z_cell ONLY
        mu, theta                     = self.dec(z_hat, c_niche, lib) # decoder sees BOTH, via FiLM
        return z_cell, q, p, p_sp, mu, theta, l_cb, l_cm

# ── losses ────────────────────────────────────────────────────────
def nb_loss(y, mu, theta, eps=1e-8):
    t1 = torch.lgamma(y+theta+eps) - torch.lgamma(theta+eps) - torch.lgamma(y+1.0)
    t2 = theta * torch.log(theta / (theta+mu+eps))
    t3 = y     * torch.log(mu    / (theta+mu+eps))
    return -(t1+t2+t3).mean()

def nb_loss_per_cell(y, mu, theta, eps=1e-8):
    t1 = torch.lgamma(y+theta+eps) - torch.lgamma(theta+eps) - torch.lgamma(y+1.0)
    t2 = theta * torch.log(theta / (theta+mu+eps))
    t3 = y     * torch.log(mu    / (theta+mu+eps))
    return -(t1+t2+t3).mean(dim=1)

# ---------------------------------------------------------------
# REMEDY 1 — gate computation  (two modes) — UNCHANGED from before
# ---------------------------------------------------------------
def compute_gate_hard(type_idx_local, ei):
    """
    type_idx_local : (n_batch,)  integer cell-type code for cells
                      CURRENTLY in the mini-batch, local order matching ei.
    ei              : (2, E)  edge_index, LOCAL indices.

    Returns w_ij in {0, 1}: exactly 1 if both endpoints share the same
    cell-type label, exactly 0 otherwise. PDF eq. (iii) — the strictest,
    most interpretable gate. No hyperparameter to tune.
    """
    ti = type_idx_local[ei[0]]
    tj = type_idx_local[ei[1]]
    return (ti == tj).float()

def compute_gate_soft(z_pca_local, ei, sigma, eps=1e-8):
    """
    z_pca_local : (n_batch, 30)  PCA embedding of the cells CURRENTLY
                  in the mini-batch, in the same local order as ei.
    ei          : (2, E)  edge_index, LOCAL indices (already remapped
                  by get_mini_batch — same convention as ew_bd).
    sigma       : fixed global bandwidth (float), computed once before
                  training from the median PCA distance across all
                  spatial edges in the full graph.

    Returns w_ij in (0, 1] for every edge: close to 1 when the two
    endpoints are transcriptomically similar (spatially-blind PCA
    space), close to 0 when they are not. Fallback for when no
    cell-type labels are available, or for direct comparison against
    the hard gate.
    """
    zi = z_pca_local[ei[0]]
    zj = z_pca_local[ei[1]]
    d2 = (zi - zj).pow(2).sum(-1)
    gate = torch.exp(-d2 / (2 * sigma ** 2 + eps))
    return gate

def loss_spatial_gated(p_sp, ei, ew, gate, eps=1e-8):
    """
    Same JSD-over-spatial-edges computation as before, except each
    edge's contribution is now multiplied by `gate` as well as `ew`.
    Uses p_sp (fixed tau=1.0) — unchanged from baseline, still prevents
    JSD inflation as tau anneals. Under Remedy 4, p_sp derives from
    z_cell distances only — see the note at the top of this cell for
    why that makes this loss doubly safe against cross-type merging.
    """
    chunk = 5000; total = 0.0; count = ei.shape[1]
    combined_w = ew * gate
    for s in range(0, count, chunk):
        e    = min(s + chunk, count)
        pi_  = p_sp[ei[0, s:e]]
        pj_  = p_sp[ei[1, s:e]]
        m    = 0.5 * (pi_ + pj_)
        jsd  = (0.5*(pi_*(torch.log(pi_+eps)-torch.log(m+eps))).sum(-1)
              + 0.5*(pj_*(torch.log(pj_+eps)-torch.log(m+eps))).sum(-1))
        total += (combined_w[s:e] * jsd).sum()
    return total / count

def loss_usage(p, eps=1e-8):
    pb = p.mean(0)
    return (pb * torch.log(pb + eps)).sum()

# ── mini-batch ────────────────────────────────────────────────────
def get_mini_batch(ei_S, ew_S, n):
    seed  = torch.randperm(n)[:B]
    hop1  = torch.unique(ei_S[1][torch.isin(ei_S[0], seed)])
    hop2  = torch.unique(ei_S[1][torch.isin(ei_S[0], hop1)])
    batch = torch.unique(torch.cat([seed, hop1, hop2]))
    mask  = torch.isin(ei_S[0], batch) & torch.isin(ei_S[1], batch)
    ei_b  = ei_S[:, mask]
    ew_b  = ew_S[mask]
    lmap  = torch.zeros(n, dtype=torch.long)
    lmap[batch] = torch.arange(len(batch))
    return batch, lmap[ei_b].to(device), ew_b.to(device)

# ── codebook maintenance ──────────────────────────────────────────
def fix_codebook(model, x, y, ei_S, ew_S, lib, n):
    model.eval()
    M = model.vq.M
    all_q, all_z, all_l = [], [], []
    with torch.no_grad():
        for s in range(0, n, 2048):
            batch = torch.arange(s, min(s+2048, n))
            mask  = torch.isin(ei_S[0], batch) & torch.isin(ei_S[1], batch)
            ei_b  = ei_S[:, mask]
            ew_b  = ew_S[mask]
            lmap  = torch.zeros(n, dtype=torch.long)
            lmap[batch] = torch.arange(len(batch))
            z_cell, q, _, _, mu, theta, _, _ = model(
                x[batch].to(device), lmap[ei_b].to(device),
                ew_b.to(device), lib[batch].to(device), tau=0.01)   # CHANGED — no lam arg
            all_q.append(q.cpu())
            all_z.append(z_cell.cpu())
            all_l.append(nb_loss_per_cell(y[batch].to(device), mu, theta).cpu())
            torch.cuda.empty_cache()
    all_q = torch.cat(all_q)
    all_z = torch.cat(all_z)
    all_l = torch.cat(all_l)
    counts = torch.bincount(all_q, minlength=M)
    dead   = torch.where(counts < N_MIN)[0].tolist()
    big    = torch.where(counts > int(2 * n / M))[0].tolist()
    for m in big:
        km = KMeans(2, n_init=1, random_state=0).fit(all_z[all_q == m].numpy())
        model.vq.codebook.data[m] = torch.tensor(
            km.cluster_centers_[0], dtype=torch.float32).to(device)
        # FIX — reset the SURVIVING entry's EMA memory too, not just its
        # position. Without this, N[m]/Sigma[m] still hold the old,
        # overcrowded running totals, and the very next batch's EMA
        # update (codebook = Sigma/N) silently snaps the position right
        # back toward the pre-split average -- undoing the split before
        # a single new cell has even arrived. This is why a mega entry
        # could keep reappearing at roughly the same size, epoch after
        # epoch, despite being "fixed" every single time.
        model.vq.N[m]     = 1.0
        model.vq.Sigma[m] = model.vq.codebook.data[m].clone()
        if dead:
            d = dead.pop(0)
            model.vq.codebook.data[d] = torch.tensor(
                km.cluster_centers_[1], dtype=torch.float32).to(device)
            model.vq.N[d] = 1.0
            model.vq.Sigma[d] = model.vq.codebook.data[d].clone()
    for m in dead:
        i      = int(all_l.argmax())
        z_star = all_z[i].to(device)
        model.vq.codebook.data[m] = z_star + torch.randn_like(z_star) * 1e-3
        model.vq.N[m]     = 1.0
        model.vq.Sigma[m] = model.vq.codebook.data[m].clone()
        all_l[i] = -float("inf")
    model.train()
    return int((counts < N_MIN).sum()), int((counts > int(3 * n / M)).sum())

# ── load data ─────────────────────────────────────────────────────
data       = torch.load(f"{SAVE_DIR}/stage4_tensors.pt", weights_only=True)
anchor_idx = np.load(f"{SAVE_DIR}/stage5_anchors.npy")
x       = data["x_input"]
y       = data["y_target"]
z_pca   = data["z_pca"]           # soft-gate fallback source
type_idx= data["cell_type_idx"]   # hard-gate source (-1 sentinel if no labels)
ei_S    = data["edge_index_S"]
ew_S    = data["edge_weight_S"]
lib     = y.sum(dim=1)
n, IN_DIM, G, M = x.shape[0], x.shape[1], y.shape[1], len(anchor_idx)
steps_per_epoch  = max(1, n // B)
print(f"n={n}  IN_DIM={IN_DIM}  G={G}  M={M}")
print(f"steps per epoch = {steps_per_epoch}")

# ---------------------------------------------------------------
# REMEDY 1 — resolve gate mode + fallback, and (if soft mode will be
# needed at all) fix the gate bandwidth sigma ONCE, globally, before
# training starts. Sigma uses the median PCA distance across ALL
# spatial edges in the full graph (same adaptive-bandwidth spirit as
# Stage 2). Fixed (not annealed, not recomputed per batch) so the
# gate's scale stays comparable across the whole run.
# ---------------------------------------------------------------
print()
has_labels = bool((type_idx >= 0).all())
if GATE_MODE == "hard" and not has_labels:
    print("REMEDY 1 — GATE_MODE='hard' requested but no cell_type labels found "
          "in stage4_tensors.pt -> falling back to GATE_MODE='soft'")
    GATE_MODE = "soft"

if GATE_MODE == "hard":
    n_classes = int(type_idx.max().item()) + 1
    print(f"REMEDY 1 — using HARD label gate  ({n_classes} cell-type classes)")
    GATE_SIGMA = None
else:
    print("REMEDY 1 — using SOFT Gaussian-kernel gate, computing bandwidth sigma ...")
    g_rng        = torch.Generator().manual_seed(42)
    sample_edges = torch.randperm(ei_S.shape[1], generator=g_rng)[:20000]
    d_sample     = (z_pca[ei_S[0, sample_edges]] - z_pca[ei_S[1, sample_edges]]).pow(2).sum(-1).sqrt()
    GATE_SIGMA   = float(d_sample.median().clamp(min=1e-6))
    print(f"  sampled {len(sample_edges):,} spatial edges")
    print(f"  GATE_SIGMA = {GATE_SIGMA:.4f}  (median PCA distance, fixed for whole run)")
print()

# ── initialise model ──────────────────────────────────────────────
model     = STAQ(IN_DIM, M, G).to(device)
params    = [p for nm, p in model.named_parameters() if "vq" not in nm]
optimizer = optim.Adam(params, lr=LR)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=15, min_lr=1e-5)

print("initialising codebook from encoder outputs at anchor cells...")
model.eval()
anchor_t = torch.tensor(anchor_idx, dtype=torch.long)
with torch.no_grad():
    hop   = torch.unique(ei_S[1][torch.isin(ei_S[0], anchor_t)])
    nodes = torch.unique(torch.cat([anchor_t, hop]))
    mask  = torch.isin(ei_S[0], nodes) & torch.isin(ei_S[1], nodes)
    ei_a  = ei_S[:, mask]
    ew_a  = ew_S[mask]
    lmap  = torch.zeros(n, dtype=torch.long)
    lmap[nodes] = torch.arange(len(nodes))
    z_a, c_a = model.enc(x[nodes].to(device), lmap[ei_a].to(device), ew_a.to(device))   # CHANGED — two outputs, only z_a needed
    E0   = z_a[lmap[anchor_t]].clone()
model.vq.initialize(E0)
del z_a, c_a, E0
torch.cuda.empty_cache()
model.train()
print("codebook initialised")

# ── training ──────────────────────────────────────────────────────
best_loss = float("inf")
history   = {
    "epoch":     [],
    "loss":      [],
    "recon":     [],
    "l_cb":      [],
    "l_cm":      [],
    "l_sp":      [],
    "l_ue":      [],
    "gate_mean": [],        # Remedy 1 diagnostic
    "gate_mode": GATE_MODE, # "hard" or "soft", constant for the run
    "architecture": "remedy4_two_branch_film",   # NEW — record which architecture produced this run
    "dead":      [],
    "mega":      [],
    "tau":       [],
}

print(f"\n{'ep':>5} {'loss':>9} {'recon':>9} {'l_cb':>8} {'l_cm':>8} "
      f"{'l_sp':>8} {'gate':>7} {'l_ue':>8} {'dead':>5} {'mega':>5} {'tau':>6}")
print("-" * 92)

for ep in range(1, EPOCHS + 1):
    tau = TAU0 * math.exp(-ep / T_ANN)
    ep_loss_list = []
    ep_recon, ep_cb, ep_cm, ep_sp, ep_ue, ep_gate = [], [], [], [], [], []
    model.train()

    for _ in range(steps_per_epoch):
        batch, ei_bd, ew_bd = get_mini_batch(ei_S, ew_S, n)
        x_b  = x[batch].to(device)
        y_b  = y[batch].to(device)
        lb_b = lib[batch].to(device)

        # REMEDY 1 — gate for this batch
        if GATE_MODE == "hard":
            type_b = type_idx[batch].to(device)
            gate   = compute_gate_hard(type_b, ei_bd)
        else:
            zpca_b = z_pca[batch].to(device)
            gate   = compute_gate_soft(zpca_b, ei_bd, GATE_SIGMA)

        optimizer.zero_grad()
        z_cell, q, p, p_sp, mu, theta, l_cb, l_cm = model(x_b, ei_bd, ew_bd, lb_b, tau)   # CHANGED — no lam arg

        l_r  = nb_loss(y_b, mu, theta)
        l_sp = loss_spatial_gated(p_sp, ei_bd, ew_bd, gate)   # << Remedy 1 >>
        l_ue = loss_usage(p)

        loss = l_r + A_CB*l_cb + A_CM*l_cm + A_SP*l_sp + A_UE*l_ue

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        torch.cuda.empty_cache()

        ep_loss_list.append(loss.item())
        ep_recon.append(l_r.item())
        ep_cb.append(l_cb.item())
        ep_cm.append(l_cm.item())
        ep_sp.append(l_sp.item())
        ep_ue.append(l_ue.item())
        ep_gate.append(gate.mean().item())

    ep_loss = sum(ep_loss_list) / len(ep_loss_list)
    avg_r   = sum(ep_recon)     / len(ep_recon)
    avg_cb  = sum(ep_cb)        / len(ep_cb)
    avg_cm  = sum(ep_cm)        / len(ep_cm)
    avg_sp  = sum(ep_sp)        / len(ep_sp)
    avg_ue  = sum(ep_ue)        / len(ep_ue)
    avg_gate= sum(ep_gate)      / len(ep_gate)

    scheduler.step(avg_r)

    n_dead, n_mega = fix_codebook(model, x, y, ei_S, ew_S, lib, n)

    if ep_loss < best_loss:
        best_loss = ep_loss
        torch.save(model.state_dict(), f"{SAVE_DIR}/staq_model.pt")

    history["epoch"].append(ep)
    history["loss"].append(ep_loss)
    history["recon"].append(avg_r)
    history["l_cb"].append(avg_cb)
    history["l_cm"].append(avg_cm)
    history["l_sp"].append(avg_sp)
    history["l_ue"].append(avg_ue)
    history["gate_mean"].append(avg_gate)
    history["dead"].append(n_dead)
    history["mega"].append(n_mega)
    history["tau"].append(tau)

    if ep % 10 == 0 or ep == 1:
        print(f"{ep:>5} {ep_loss:>9.4f} {avg_r:>9.4f} {avg_cb:>8.4f} "
              f"{avg_cm:>8.4f} {avg_sp:>8.4f} {avg_gate:>7.4f} {avg_ue:>8.4f} "
              f"{n_dead:>5} {n_mega:>5} {tau:>6.3f}")

np.save(f"{SAVE_DIR}/staq_history.npy", history)
torch.save(model.state_dict(), f"{SAVE_DIR}/staq_model.pt")

print("running final inference pass...")
model.eval()
all_q2, all_z2, all_p2 = [], [], []
with torch.no_grad():
    for s in range(0, n, 2048):
        batch = torch.arange(s, min(s+2048, n))
        mask  = torch.isin(ei_S[0], batch) & torch.isin(ei_S[1], batch)
        ei_b  = ei_S[:, mask]; ew_b = ew_S[mask]
        lmap  = torch.zeros(n, dtype=torch.long)
        lmap[batch] = torch.arange(len(batch))
        z_cell2, q2, p2, _, _, _, _, _ = model(
            x[batch].to(device), lmap[ei_b].to(device),
            ew_b.to(device), lib[batch].to(device), tau=0.01)   # CHANGED — no lam arg
        all_q2.append(q2.cpu()); all_z2.append(z_cell2.cpu()); all_p2.append(p2.cpu())
        torch.cuda.empty_cache()
final_q = torch.cat(all_q2)
final_z = torch.cat(all_z2)
final_p = torch.cat(all_p2)
cb      = model.vq.codebook.detach().cpu()

torch.save(final_q, f"{SAVE_DIR}/stage6_assignments.pt")
torch.save(final_z, f"{SAVE_DIR}/stage6_embeddings.pt")
torch.save(final_p, f"{SAVE_DIR}/stage6_soft_assignments.pt")
torch.save(cb,      f"{SAVE_DIR}/stage6_codebook.pt")
np.save(f"{SAVE_DIR}/stage6_history.npy", history)
print(f"unique metacells used: {final_q.unique().shape[0]}")
print(f"final gate mean       : {history['gate_mean'][-1]:.4f}")
print("[SUCCESS] all stage6 outputs saved")

print(f"\nbest loss : {best_loss:.4f}")
print("[SUCCESS] staq_model.pt and staq_history.npy saved")


## Diagnostic — UMAP of the encoder's own z_cell embeddings

Before the codebook or Stage 7 ever touch anything, this checks the
encoder itself directly: does `z_cell` — the spatially blind branch
that alone feeds the codebook — actually separate cell types well on
its own? Similar-type cells should cluster together here; if they
don't, that points at the encoder itself, upstream of the codebook,
rather than at the assignment or refinement steps.

In [ ]:
# =================================================================
# DIAGNOSTIC — UMAP of z_cell embeddings, coloured by true cell type
# =================================================================
import os
import numpy as np
import torch
import scanpy as sc

SAVE_DIR = "/kaggle/working"
CT_COL   = "cell_type"

z_cell_saved = torch.load(os.path.join(SAVE_DIR, "stage6_embeddings.pt"), weights_only=True).numpy()
adata_check  = sc.read_h5ad(os.path.join(SAVE_DIR, "stage1.h5ad"))

print(f"z_cell embeddings shape: {z_cell_saved.shape}")

emb_adata = sc.AnnData(z_cell_saved.astype("float32"))
emb_adata.obs[CT_COL] = adata_check.obs[CT_COL].values
emb_adata.obs[CT_COL] = emb_adata.obs[CT_COL].astype("category")

# z_cell is already a small, dense learned embedding (D=32) -- use it
# directly for neighbours, no PCA on top of it needed
sc.pp.neighbors(emb_adata, use_rep="X")
sc.tl.umap(emb_adata)
sc.pl.umap(emb_adata, color=CT_COL,
           title="UMAP of z_cell embeddings (Stage 6) — coloured by true cell type")


## Loss curves (5 losses + total + Remedy 1 gate diagnostic)

In [ ]:
# =================================================================
# LOSS CURVES  (5 losses + total + Remedy 1 gate diagnostic)
# =================================================================

import os
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

matplotlib.rcParams.update({
    "figure.dpi":        150,
    "font.family":       "sans-serif",
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.titlesize":    11,
    "axes.labelsize":    10,
    "xtick.labelsize":   9,
    "ytick.labelsize":   9,
    "legend.fontsize":   9,
})

SAVE_DIR = "/kaggle/working"
hist     = np.load(f"{SAVE_DIR}/staq_history.npy", allow_pickle=True).item()
epochs   = hist["epoch"]

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
fig.suptitle("STAQ — Loss Curves (Remedy 1: gated spatial loss)",
             fontsize=13, fontweight="bold")

ax = axes[0, 0]
ax.plot(epochs, hist["recon"], color="#16a34a", lw=2)
ax.set_title("L_recon (NB neg. log-lik.)\nshould decrease")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.set_yscale("log")
best_r = min(hist["recon"])
ax.axhline(best_r, color="#16a34a", ls=":", lw=1.5, label=f"best = {best_r:.4f}")
ax.legend()

ax = axes[0, 1]
ax.plot(epochs, hist["l_cb"], color="#2563eb", lw=2)
ax.set_title("L_codebook\nshould stay small")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
best_cb = min(hist["l_cb"])
ax.axhline(best_cb, color="#2563eb", ls=":", lw=1.5, label=f"best = {best_cb:.4f}")
ax.legend()

ax = axes[0, 2]
ax.plot(epochs, hist["l_cm"], color="#dc2626", lw=2)
ax.set_title("L_commit\nshould stay small")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
best_cm = min(hist["l_cm"])
ax.axhline(best_cm, color="#dc2626", ls=":", lw=1.5, label=f"best = {best_cm:.4f}")
ax.legend()

ax = axes[0, 3]
ax.plot(epochs, hist["l_ue"], color="#7c3aed", lw=2)
ax.set_title("L_usage  -H(p_bar)\nshould decrease")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
best_ue = min(hist["l_ue"])
ax.axhline(best_ue, color="#7c3aed", ls=":", lw=1.5, label=f"best = {best_ue:.4f}")
ax.legend()

ax = axes[1, 0]
ax.plot(epochs, hist["l_sp"], color="#ea580c", lw=2)
ax.set_title("L_spatial — GATED JSD\n(Remedy 1) should be small & stable")
ax.set_xlabel("Epoch"); ax.set_ylabel("JSD")
best_sp = min(hist["l_sp"])
ax.axhline(best_sp, color="#ea580c", ls=":", lw=1.5, label=f"best = {best_sp:.4f}")
ax.legend()

ax = axes[1, 1]
ax.plot(epochs, hist["gate_mean"], color="#0891b2", lw=2)
gate_mode_label = hist.get("gate_mode", "?")
ax.set_title(f"Remedy 1 — mean gate w_ij  (mode='{gate_mode_label}')\nlow = many cross-type edges suppressed")
ax.set_xlabel("Epoch"); ax.set_ylabel("mean w_ij")
ax.set_ylim(0, 1)
avg_gate = float(np.mean(hist["gate_mean"]))
ax.axhline(avg_gate, color="#0891b2", ls=":", lw=1.5, label=f"avg = {avg_gate:.4f}")
ax.legend()

ax = axes[1, 2]
ax.plot(epochs, hist["loss"], color="#0f172a", lw=2)
ax.set_title("Total Loss")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.set_yscale("log")
best_t = min(hist["loss"])
ax.axhline(best_t, color="#0f172a", ls=":", lw=1.5, label=f"best = {best_t:.4f}")
ax.legend()

ax = axes[1, 3]
ax.plot(epochs, hist["dead"], color="#b45309", lw=1.5, label="dead codes")
ax.plot(epochs, hist["mega"], color="#7c2d12", lw=1.5, label="mega codes")
ax.set_title("Codebook health\n(dead / oversized entries per epoch)")
ax.set_xlabel("Epoch"); ax.set_ylabel("count")
ax.legend()

for a in axes.flat:
    a.set_facecolor("#fafafa")

plt.tight_layout()
os.makedirs(os.path.join(SAVE_DIR, "figures"), exist_ok=True)
plt.savefig(os.path.join(SAVE_DIR, "figures", "plot_staq_losses.png"),
            dpi=200, bbox_inches="tight")
plt.show()

print("saved: figures/plot_staq_losses.png")
print()
print(f"best recon    : {best_r:.4f}")
print(f"best codebook : {best_cb:.4f}")
print(f"best commit   : {best_cm:.4f}")
print(f"best spatial  : {best_sp:.4f}   (gated)")
print(f"best usage    : {best_ue:.4f}")
print(f"best total    : {best_t:.4f}")
print(f"mean gate w_ij (whole run) : {avg_gate:.4f}")


## Stage 8 — Aggregation (Stage 7 contiguity refinement removed)

Reads Stage 6's hard assignment directly — no spatial contiguity
refinement in between. A metacell here is purely "cells that landed
on the same codebook entry," with no guarantee they're physically
adjacent on the tissue. Also now writes the `metacell` column back
into `stage1.h5ad` itself, since Stage 7 used to be the only place
doing that.

In [ ]:
# =================================================================
# STAGE 8 — AGGREGATION  (Stage 7 removed — using Stage 6 assignment directly)
# =================================================================
# Computes pseudobulk Y, centroid S_bar, size per metacell; saves the
# complete metacell profile object to metacells.h5ad. Also now writes
# the "metacell" column back into stage1.h5ad itself, since Stage 7
# (deleted) was previously the only place doing that, and the plots
# cell depends on it being there.
# =================================================================

import os
import numpy as np
import torch
import scanpy as sc
import anndata

SAVE_DIR = "/kaggle/working"
print("=" * 60)
print("STAGE 8 — Aggregation")
print("=" * 60)

data   = torch.load(os.path.join(SAVE_DIR, "stage4_tensors.pt"), weights_only=True)
pi     = torch.load(os.path.join(SAVE_DIR, "stage6_assignments.pt"), weights_only=True).numpy()   # CHANGED — Stage 7 removed, use Stage 6 hard assignment directly
adata  = sc.read_h5ad(os.path.join(SAVE_DIR, "stage1.h5ad"))

y_raw  = data["y_target"].numpy()
coords = adata.obsm["spatial"].astype("float32")

n, g = y_raw.shape; ds = coords.shape[1]; M = int(pi.max()) + 1
print(f"n={n}   g={g}   M={M}")

Y     = np.zeros((M, g),  dtype=np.float32)
S_bar = np.zeros((M, ds), dtype=np.float32)
sizes = np.zeros(M,       dtype=np.int64)

for m in range(M):
    idx = np.where(pi == m)[0]
    if len(idx) == 0: continue
    sizes[m] = len(idx)
    Y[m]     = y_raw[idx].sum(axis=0)
    S_bar[m] = coords[idx].mean(axis=0)

occ = (sizes > 0)
print(f"occupied metacells : {occ.sum()} / {M}")
print(f"size distribution  : min={sizes[occ].min()}  "
      f"median={int(np.median(sizes[occ]))}  "
      f"max={sizes.max()}"
      f"mean={sizes[occ].mean():.1f}")
counts_match = np.isclose(Y.sum(), y_raw.sum())
print(f"counts sanity      : Y.sum()={Y.sum():.0f}  raw.sum()={y_raw.sum():.0f}  "
      f"match={counts_match}  {'OK' if counts_match else 'BUG!'}")

np.save(os.path.join(SAVE_DIR, "stage8_Y.npy"),     Y)
np.save(os.path.join(SAVE_DIR, "stage8_S_bar.npy"), S_bar)
np.save(os.path.join(SAVE_DIR, "stage8_sizes.npy"), sizes)

print()
print("OUTPUT:")
print(f"  stage8_Y.npy      shape=({M},{g})   pseudobulk raw counts")
print(f"  stage8_S_bar.npy  shape=({M},{ds})     centroid coordinates")
print(f"  stage8_sizes.npy  shape=({M},)     cells per metacell")
print()
print("[SUCCESS] stage8 saved")

# NEW — write the metacell assignment back into stage1.h5ad (Stage 7 used to do this)
adata.obs["metacell"] = pi.astype(str)
adata.obs["metacell"] = adata.obs["metacell"].astype("category")
adata.write_h5ad(os.path.join(SAVE_DIR, "stage1.h5ad"))
print("metacell assignment saved to stage1.h5ad")

# ── save complete metacell profile object ────────────────────────
mc_adata = anndata.AnnData(X=Y.astype(np.float32))
mc_adata.obsm["spatial"]    = S_bar
mc_adata.obs["size"]        = sizes[sizes > 0] if (sizes > 0).sum() == Y.shape[0] else sizes[:Y.shape[0]]
mc_adata.obs["metacell_id"] = np.arange(M)
mc_adata.var_names          = adata.var_names

CT_COL = "cell_type"
if CT_COL in adata.obs.columns:
    cell_types   = adata.obs[CT_COL].astype(str).values
    unique_types = np.unique(cell_types)
    dom_type, modal_purity = [], []
    for m in range(M):
        idx = np.where(pi == m)[0]
        if len(idx) == 0:
            dom_type.append("unknown"); modal_purity.append(np.nan); continue
        types_here = cell_types[idx]
        counts_ct  = {t: (types_here == t).sum() for t in unique_types
                      if (types_here == t).sum() > 0}
        best_t = max(counts_ct, key=counts_ct.get)
        dom_type.append(best_t)
        modal_purity.append(counts_ct[best_t] / len(idx))
    mc_adata.obs[CT_COL]        = dom_type
    mc_adata.obs[CT_COL]        = mc_adata.obs[CT_COL].astype("category")
    mc_adata.obs["modal_purity"] = modal_purity
    print(f"dominant {CT_COL} + modal_purity added to metacells.h5ad")

mc_adata.write_h5ad(os.path.join(SAVE_DIR, "metacells.h5ad"))
print(f"metacells.h5ad saved  — shape: {mc_adata.shape}")
print(f"  X           : pseudobulk raw counts  {mc_adata.X.shape}")
print(f"  obsm spatial: centroid coordinates    {mc_adata.obsm['spatial'].shape}")
print(f"  obs size    : cells per metacell      min={mc_adata.obs['size'].min()}  max={mc_adata.obs['size'].max()}")


## Stage 9 — Evaluation metrics

Unchanged core metrics from the pipeline (kappa_T, kappa_S, niche entropy,
inner connectedness, quantisation gap, codebook usage balance), plus one
addition directly relevant to judging whether Remedy 1 worked: **modal-type
purity** (already computed per metacell in Stage 8, summarised here
alongside the rest of the metric suite).

In [ ]:
# =================================================================
# STAGE 9 — EVALUATION METRICS
# =================================================================

import os, math
import numpy as np
import torch
import scanpy as sc
from collections import deque
import warnings
warnings.filterwarnings("ignore")

SAVE_DIR = "/kaggle/working"
print("=" * 60)
print("STAGE 9 — Evaluation Metrics")
print("=" * 60)

adata    = sc.read_h5ad(os.path.join(SAVE_DIR, "stage1.h5ad"))
mc_adata = sc.read_h5ad(os.path.join(SAVE_DIR, "metacells.h5ad"))
data     = torch.load(os.path.join(SAVE_DIR, "stage4_tensors.pt"), weights_only=True)
pi       = torch.load(os.path.join(SAVE_DIR, "stage6_assignments.pt"), weights_only=True).numpy()   # CHANGED — Stage 7 removed
sizes    = np.load(os.path.join(SAVE_DIR, "stage8_sizes.npy"))
all_z    = torch.load(os.path.join(SAVE_DIR, "stage6_embeddings.pt"), weights_only=True).numpy()
all_p    = torch.load(os.path.join(SAVE_DIR, "stage6_soft_assignments.pt"), weights_only=True)
codebook = torch.load(os.path.join(SAVE_DIR, "stage6_codebook.pt"), weights_only=True).numpy()
M_orig   = len(np.load(os.path.join(SAVE_DIR, "stage5_anchors.npy")))
hist     = np.load(os.path.join(SAVE_DIR, "staq_history.npy"), allow_pickle=True).item()
ei_S     = data["edge_index_S"]

Z      = adata.obsm["X_pca"].astype("float32")
coords = adata.obsm["spatial"].astype("float32")
n      = len(pi); M = int(pi.max()) + 1; occ = sizes > 0
print(f"n={n}   M={M}   M_orig={M_orig}")

nb_list = [[] for _ in range(n)]
for s_, d_ in zip(ei_S[0].numpy(), ei_S[1].numpy()):
    nb_list[s_].append(int(d_))

def bfs_components(cell_list):
    cs, vis, comps = set(cell_list), set(), []
    for start in cell_list:
        if start in vis: continue
        comp, q = [], deque([start]); vis.add(start)
        while q:
            node = q.popleft(); comp.append(node)
            for nb in nb_list[node]:
                if nb in cs and nb not in vis: vis.add(nb); q.append(nb)
        comps.append(comp)
    return comps

# ── [1] transcriptomic compactness kappa_T ───────────────────────
print("\n[1] Transcriptomic compactness kappa_T ...")
kT = np.zeros(M)
for m in range(M):
    idx = np.where(pi == m)[0]
    if len(idx) < 2: continue
    centroid = Z[idx].mean(axis=0)
    kT[m]    = float(np.median(np.linalg.norm(Z[idx] - centroid, axis=1)))
print(f"    mean={kT[occ].mean():.4f}   median={np.median(kT[occ]):.4f}   max={kT[occ].max():.4f}")

# ── [2] spatial compactness kappa_S ──────────────────────────────
print("\n[2] Spatial compactness kappa_S ...")
kS = np.zeros(M)
for m in range(M):
    idx = np.where(pi == m)[0]
    if len(idx) < 2: continue
    centroid = coords[idx].mean(axis=0)
    kS[m]    = float(np.median(np.linalg.norm(coords[idx] - centroid, axis=1)))
print(f"    mean={kS[occ].mean():.4f}   median={np.median(kS[occ]):.4f}   max={kS[occ].max():.4f}")

# ── [3] niche entropy + modal purity ─────────────────────────────
print("\n[3] Niche entropy + modal-type purity ...")
niche_entropy = None; purity_arr = None; ct_col = None
for col in ["cell_type","celltype","CellType","cell_class",
            "subclass","cluster","leiden","louvain"]:
    if col in adata.obs.columns: ct_col = col; break
if ct_col:
    print(f"    using adata.obs['{ct_col}']")
    labels = adata.obs[ct_col].values
    niche_entropy = np.zeros(M)
    purity_arr    = np.zeros(M)
    for m in range(M):
        idx = np.where(pi == m)[0]
        if len(idx) == 0: continue
        counts = {}
        for c in labels[idx]: counts[c] = counts.get(c, 0) + 1
        total = len(idx)
        niche_entropy[m] = -sum((v/total)*math.log(v/total+1e-12) for v in counts.values())
        purity_arr[m]    = max(counts.values()) / total
    n_types = len(set(labels))
    print(f"    niche entropy : mean={niche_entropy[occ].mean():.4f}   "
          f"max_possible=log({n_types})={math.log(n_types):.3f}")
    print(f"    modal purity  : mean={purity_arr[occ].mean():.4f}   "
          f"median={np.median(purity_arr[occ]):.4f}   [higher = purer]")
else:
    print(f"    SKIPPED — no cell-type column found")

# ── [4] purity vs ground truth ───────────────────────────────────
print("\n[4] Purity vs simulated ground truth — SKIPPED (none available)")

# ── [5] inner connectedness ──────────────────────────────────────
print("\n[5] Inner connectedness ...")
n_single = 0; n_occ_cc = 0; n_comps = np.zeros(M, dtype=int)
for m in range(M):
    idx = np.where(pi == m)[0].tolist()
    if not idx: continue
    n_occ_cc += 1
    comps = bfs_components(idx); n_comps[m] = len(comps)
    if len(comps) == 1: n_single += 1
ic = n_single / n_occ_cc if n_occ_cc else 0.0
n_frag = int((n_comps[occ] > 1).sum())
print(f"    inner connectedness = {ic:.4f}   ({n_single}/{n_occ_cc} contiguous)   [target > 0.95]")
print(f"    fragmented metacells = {n_frag}")

# ── [6] quantisation gap ─────────────────────────────────────────
print("\n[6] Quantisation gap ...")
valid      = pi < M_orig
mean_gap   = float(np.sum((all_z[valid] - codebook[pi[valid]])**2, axis=1).mean())
rng        = np.random.default_rng(42)
ii         = rng.integers(0, M_orig, 5000); jj = rng.integers(0, M_orig, 5000)
jj[ii==jj] = (jj[ii==jj] + 1) % M_orig
med_inter  = float(np.median(np.linalg.norm(codebook[ii] - codebook[jj], axis=1)))
quant_gap  = mean_gap / (med_inter + 1e-8)
print(f"    mean ||z-e||^2 = {mean_gap:.4f}   median inter-cb = {med_inter:.4f}   "
      f"gap = {quant_gap:.4f}   [target < 0.50]")

# ── [7] codebook usage balance ───────────────────────────────────
print("\n[7] Codebook usage balance ...")
p_bar   = all_p.mean(dim=0)
H_usage = -(p_bar * torch.log(p_bar + 1e-12)).sum().item()
H_max   = math.log(M_orig)
balance = H_usage / H_max
print(f"    H(p_bar)={H_usage:.4f}   log(M)={H_max:.4f}   balance={balance:.4f}   [target > 0.80]")

# ── Remedy 1 diagnostic ───────────────────────────────────────────
avg_gate_run = float(np.mean(hist["gate_mean"]))
gate_mode_run = hist.get("gate_mode", "unknown")
print(f"\n[Remedy 1] gate mode: {gate_mode_run}   mean gate w_ij across training: {avg_gate_run:.4f}")

print("\n" + "=" * 52)
print("  STAQ — Evaluation Metrics Summary")
print("=" * 52)
print(f"  [1] kappa_T transcriptomic compactness : {kT[occ].mean():.4f}")
print(f"  [2] kappa_S spatial compactness         : {kS[occ].mean():.4f}")
if niche_entropy is not None:
    print(f"  [3]     niche entropy              : {niche_entropy[occ].mean():.4f}")
    print(f"  [3b]    modal-type purity          : {purity_arr[occ].mean():.4f}")
else:
    print(f"  [3]     niche entropy              : N/A")
print(f"  [4]     purity vs ground truth      : SKIPPED")
print(f"  [5]     inner connectedness         : {ic:.4f}   [> 0.95]")
print(f"  [6]     quantisation gap            : {quant_gap:.4f}   [< 0.50]")
print(f"  [7]     codebook usage balance      : {balance:.4f}   [> 0.80]")
print(f"  [R1]    gate mode                   : {gate_mode_run}")
print(f"  [R1]    mean gate w_ij              : {avg_gate_run:.4f}")
print("=" * 52)

out = dict(kT=kT, kS=kS, n_components=n_comps,
           inner_connectedness=np.array([ic]),
           quant_gap=np.array([quant_gap]),
           usage_balance=np.array([balance]),
           metacell_sizes=sizes,
           mean_gate=np.array([avg_gate_run]),
           gate_mode=np.array([gate_mode_run]))
if niche_entropy is not None:
    out["niche_entropy"] = niche_entropy
    out["modal_purity"]  = purity_arr
np.savez(os.path.join(SAVE_DIR, "stage9_metrics.npz"), **out)
print()
print("[SUCCESS] stage9_metrics.npz saved")


## Results plots — simple, scanpy-only, consistent

Spatial by cell type, spatial by metacell, UMAP by cell type, UMAP by
metacell, a spatial plot with original cells as background and
metacell centroids overlaid on top (fixed size, not scaled by how
many cells each metacell has — just enough to tell them apart from
the background), cells-per-metacell, purity distribution, and the
average purity number.

In [ ]:
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt

sc.settings.set_figure_params(dpi=100, figsize=(6, 6), facecolor="white")
CT_COL = "cell_type"

adata    = sc.read_h5ad("stage1.h5ad")
mc_adata = sc.read_h5ad("metacells.h5ad")
adata.obs[CT_COL]     = adata.obs[CT_COL].astype("category")
adata.obs["metacell"] = adata.obs["metacell"].astype("category")
mc_adata.obs[CT_COL]  = mc_adata.obs[CT_COL].astype("category")

M = mc_adata.n_obs
print(f"n={adata.n_obs} cells   M={M} metacells")

# 1 — spatial, coloured by cell type
sc.pl.embedding(adata, basis="spatial", color=CT_COL, title="Spatial — cell type")

# 2 — spatial, coloured by metacell
sc.pl.embedding(adata, basis="spatial", color="metacell", legend_loc=None,
                 title="Spatial — metacell assignment")

# 3 & 4 — UMAP, coloured by cell type / metacell
sc.pp.neighbors(adata, use_rep="X_pca")
sc.tl.umap(adata)
sc.pl.umap(adata, color=CT_COL, title="UMAP — cell type")
sc.pl.umap(adata, color="metacell", legend_loc=None, title="UMAP — metacell assignment")

# 5 — spatial: original cells (background) + metacell centroids overlaid,
# fixed marker size (NOT scaled by metacell size) just enough to tell
# them apart from the background. Both adata and mc_adata already share
# the same real spatial coordinate system, so no projection is needed --
# the centroids are plotted directly.
fig, ax = plt.subplots(figsize=(7, 7))
sc.pl.embedding(adata, basis="spatial", color=CT_COL, ax=ax, show=False,
                 size=15, alpha=0.5, legend_loc=None, title="")
type_colors = dict(zip(adata.obs[CT_COL].cat.categories, adata.uns[f"{CT_COL}_colors"]))
mc_dot_colors = mc_adata.obs[CT_COL].map(type_colors)
ax.scatter(mc_adata.obsm["spatial"][:, 0], mc_adata.obsm["spatial"][:, 1],
           s=90, c=mc_dot_colors, edgecolors="black", linewidths=1.3, zorder=5)
ax.set_title("Spatial — original cells with metacell centroids overlaid")
plt.show()

# 6 — cells per metacell
sizes = mc_adata.obs["size"].values
order = np.argsort(-sizes)
plt.figure(figsize=(8, 3))
plt.bar(range(M), sizes[order], color="#4b5563")
plt.xlabel("metacell (sorted by size)"); plt.ylabel("cells assigned")
plt.title("Cells per metacell")
plt.show()

# 7 — purity distribution
purity = mc_adata.obs["modal_purity"].values
plt.figure(figsize=(6, 4))
plt.hist(purity, bins=20, range=(0, 1), color="#7c3aed")
plt.xlabel("purity"); plt.ylabel("count")
plt.title("Metacell purity distribution")
plt.show()

# 8 — average purity
print(f"Average purity: {np.nanmean(purity):.3f}")
